# GameForge3D — Phase 1: Data Collection & Preparation

**FYP 2026-2027 | NUML Dept. of Computer Science**  
**Supervisor:** Ms. Tooba Sagheer

This notebook handles:
1. Google Drive mount & checkpoint setup
2. Library installation
3. Cap3D dataset download from HuggingFace
4. Cap3D dataset exploration & gaming subset filtering
5. Asset Router CSV dataset upload & validation
6. Data split (train / val / test)


## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/GameForge3D/checkpoints'
DATA_DIR       = '/content/drive/MyDrive/GameForge3D/data'

os.makedirs(f'{CHECKPOINT_DIR}/router',    exist_ok=True)
os.makedirs(f'{CHECKPOINT_DIR}/generator', exist_ok=True)
os.makedirs(f'{CHECKPOINT_DIR}/texture',   exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print('Drive mounted.')
print(f'Checkpoint dir : {CHECKPOINT_DIR}')
print(f'Data dir       : {DATA_DIR}')

## Step 2 — Install Libraries

In [ ]:
!pip install -q torch torchvision transformers diffusers \
               open3d pymeshlab trimesh reportlab qrcode \
               fastapi uvicorn datasets pandas scikit-learn \
               matplotlib seaborn
print('All libraries installed.')

## Step 3 — Load Cap3D Dataset from HuggingFace

Cap3D provides **785,000+** (text caption, 3D point-cloud) pairs.  
We will:
- Stream the dataset (no full download needed)
- Preview the first few samples
- Filter for gaming-relevant objects

In [ ]:
from datasets import load_dataset
import pandas as pd

print('Loading Cap3D dataset from HuggingFace (streaming)...')
# streaming=True means no full download — reads on-the-fly
cap3d = load_dataset('tiange/Cap3D', split='test', streaming=True)

# Preview first 5 samples
samples = []
for i, item in enumerate(cap3d):
    samples.append(item)
    if i >= 4:
        break

df_preview = pd.DataFrame(samples)
print(f'Columns: {df_preview.columns.tolist()}')
print(df_preview.head())

## Step 4 — Filter Gaming-Relevant Captions from Cap3D

We keep captions that contain keywords related to our 4 categories:  
`Weapon`, `Vehicle`, `Prop`, `Creature`

In [ ]:
WEAPON_KW   = ['sword','axe','bow','gun','rifle','blade','dagger','spear',
               'lance','mace','cannon','pistol','crossbow','wand','staff',
               'knife','whip','shield','grenade','hammer']

VEHICLE_KW  = ['car','truck','bike','motorcycle','ship','boat','plane',
               'aircraft','submarine','tank','helicopter','spacecraft',
               'rover','chariot','vessel','train','bus','jet']

CREATURE_KW = ['dragon','monster','creature','beast','wolf','zombie',
               'goblin','demon','ghost','skeleton','giant','fairy',
               'troll','vampire','golem','robot','alien','elemental']

PROP_KW     = ['barrel','chest','crate','lantern','bottle','table',
               'chair','door','fountain','altar','torch','key',
               'treasure','scroll','statue','pillar','ruin','cage']

ALL_KW = WEAPON_KW + VEHICLE_KW + CREATURE_KW + PROP_KW

print('Filtering Cap3D for gaming-relevant captions (scanning 50,000 samples)...')
gaming_samples = []
cap3d_fresh    = load_dataset('tiange/Cap3D', split='test', streaming=True)

for i, item in enumerate(cap3d_fresh):
    caption = item.get('caption', '').lower()
    if any(kw in caption for kw in ALL_KW):
        gaming_samples.append({'uid': item.get('uid',''), 'caption': item.get('caption','')})
    if i >= 50000:
        break

df_gaming = pd.DataFrame(gaming_samples)
print(f'Gaming-relevant samples found: {len(df_gaming)}')
df_gaming.to_csv(f'{DATA_DIR}/cap3d_gaming_subset.csv', index=False)
print(f'Saved to {DATA_DIR}/cap3d_gaming_subset.csv')
df_gaming.head(10)

## Step 5 — Upload & Validate Asset Router CSV

Upload `router_labels.csv` from the GitHub repo (already committed).  
Or upload manually using the file dialog below.

In [ ]:
# Option A: Clone from GitHub (recommended)
import subprocess, os

if not os.path.exists('/content/GameForge3D'):
    subprocess.run(['git', 'clone',
                    'https://github.com/Zubair-471/GameForge3D.git',
                    '/content/GameForge3D'], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', '/content/GameForge3D', 'pull'], check=True)
    print('Repo updated.')

ROUTER_CSV = '/content/GameForge3D/data/router_labels.csv'
df_router  = pd.read_csv(ROUTER_CSV)
print(f'\nRouter dataset shape: {df_router.shape}')
print(df_router['label'].value_counts())

## Step 6 — Train / Val / Test Split

Split router dataset into **70% train, 15% val, 15% test**.

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 85% train+val, 15% test
df_trainval, df_test = train_test_split(
    df_router, test_size=0.15, random_state=42, stratify=df_router['label'])

# Second split: 70% train, 15% val (from 85%)
df_train, df_val = train_test_split(
    df_trainval, test_size=0.176, random_state=42, stratify=df_trainval['label'])

print(f'Train : {len(df_train)} samples')
print(f'Val   : {len(df_val)} samples')
print(f'Test  : {len(df_test)} samples')

# Save splits
df_train.to_csv(f'{DATA_DIR}/router_train.csv', index=False)
df_val.to_csv(f'{DATA_DIR}/router_val.csv',   index=False)
df_test.to_csv(f'{DATA_DIR}/router_test.csv',  index=False)

print(f'\nSplits saved to {DATA_DIR}')

## Step 7 — Label Distribution Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
splits = {'Train': df_train, 'Val': df_val, 'Test': df_test}

for ax, (name, df) in zip(axes, splits.items()):
    counts = df['label'].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='viridis')
    ax.set_title(f'{name} Split ({len(df)} samples)')
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.5, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/label_distribution.png', dpi=150)
plt.show()
print('Label distribution chart saved.')

## ✅ Phase 1 Complete!

| Output | Location |
|--------|----------|
| Cap3D Gaming Subset | `GameForge3D/data/cap3d_gaming_subset.csv` |
| Router Train Split | `GameForge3D/data/router_train.csv` |
| Router Val Split | `GameForge3D/data/router_val.csv` |
| Router Test Split | `GameForge3D/data/router_test.csv` |
| Label Distribution Chart | `GameForge3D/data/label_distribution.png` |

**Next Step → Phase 2: Asset Category Router (DistilBERT fine-tuning)**